In [ ]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/pipeline_handler.py
import sys
import json
import math
import logging
from typing import Any, Dict, List, Optional, Union
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from qdrant_client import QdrantClient
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_ollama import ChatOllama
from src.config import (
    DEFAULT_KEEP_ALIVE,
    DEFAULT_NUM_THREAD,
    LLM_MODEL,
    OLLAMA_URL,
)

from src.config import (
    QDRANT_URL, 
    COLLECTION_NAME, 
    EMBEDDING_MODEL, 
    RETRIEVAL_TOP_K,
    RERANKING_TOP_N
)

from src.sidecar_manager import SidecarManager
from src.graph_builder import KnowledgeGraphBuilder
from src.chunk_widget import ChunkGraphWidget
from src.tag_assigner import TagAssigner

from src.distance_retriever import DistanceRetriever, build_query_tags
from src.distance_tag_reranker import DistanceTagReranker

# [DEBUG: Logger]
logger = logging.getLogger(__name__)

class PipelineHandler:
    """
    Gestore centralizzato della pipeline graphRAG implementata.
    Offre i metodi per le varie fasi e la gestione e modifica dei parametri utilizzati
    per poter effettuare le necessarie prove.
    """

    ### Inizializzazione moduli
    def __init__(
        self,
        sidecar_path: str=None,
        queries_path: Optional[Union[str, Path]] = None
    ):
        # Inizializzazione sidecar_manager
        self.sidecar_path = sidecar_path
        
        self.sidecar_manager = None 
        if not self.sidecar_path:
            self.sidecar_manager=SidecarManager()
        else:
            self.sidecar_manager=SidecarManager(filepath=self.sidecar_path)

        # Inizializzazione Vector Store con FastEmbed nativo
        self.embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)
        self.qdrant_client = QdrantClient(url=QDRANT_URL)
        
        # Inizializzazione llama
        self.llm = ChatOllama(
            model=LLM_MODEL,
            base_url=OLLAMA_URL,
            keep_alive=DEFAULT_KEEP_ALIVE,
            num_thread=DEFAULT_NUM_THREAD,
            temperature=0,
        )

        # Inizializzazione widget
        self.widget = ChunkGraphWidget(sidecar=self.sidecar_manager)
        
        # Inizializzazione moduli gestinoe Tag
        self.tag_assigner = TagAssigner()
        
        # Inizializzazione Reranker
        self.dt_reranker = DistanceTagReranker(qdrant_client=self.qdrant_client, sidecar_manager=self.sidecar_manager)

        # Parametri query
        self.query_text = ""
        self.queries_path = queries_path
        self.query_vector = None
        
        self.conversation_tags = []
        self.assigned_tags = []
        self.automatic_tags = []
        self.auto_tag_assignation = False
        self.query_tags = {}

        # Parametri retrieval e reranking
        self.personalised_top_n = RERANKING_TOP_N
        self.personalised_top_k = RETRIEVAL_TOP_K
        self.personalised_top_j = self.personalised_top_k
        
        logger.info("!>> Pipeline Inizializzata")

    ### Gestione Query
    def update_query(
        self,
        new_text = Optional[str]=None,
        new_id = Optional[str]=None,
    ):
        """
        Aggiorna il testo della query e il suo embedding.
        Se forniti entrambi i parametri, da la priorità all'ID.
        """
        if new_id:
            if not self.queries_path:
                logger.warning("!!!> update_query(...) -> OPERAZIONE ABORTITA: Fornito un QUERY_ID ma manca il QUERY_PATH.")
                return
            queries_path = self.queries_path
            with open(queries_path, "r", encoding="utf-8") as f:
                benchmark_queries = json.load(f)
            
            matched = [q for q in benchmark_queries if q.get("id") == new_id]
            query_obj = matched[0]
            self.query_text = query_obj["query"]
            logger.info(f"?> Testo Query aggiornato: {self.queries.path}--ID>{new_id} ")
        elif new_text:
            self.query_text = new_text
            logger.info(f"?> Testo Query aggiornato con il testo fornito ")
        else:
            logger.warning(f"!> update_query(...) -> Invocazione senza alterazioni")
        
        self.query_vector = self.embeddings.embed_query(self.query_text)
        logger.info(f"?> Vettore embedding Query aggiornato: text:{self.query_text}")

    def print_current_global_tags():
        """
        Stampa i tag globali attualmente in uso.
        """
        sidecar_data = self.sidecar_manager.load_data()
        global_tags_dict = sidecar_data.get("global_tags", {})
        global_tags = list(global_tags_dict.keys())
        print(f">> Testo Query: '{self.query_text}'\n")
        print(f">> Tag Globali: {global_tags}")

    def update_query_tags(
        self,
        add_conv_tag = Optional[str]=None,
        rmv_conv_tag = Optional[str]=None,
        new_conv_list = Optional[List[str]]=None
        add_ass_tag = Optional[str]=None,
        rmv_ass_tag = Optional[str]=None,
        new_ass_list = Optional[List[str]]=None
        auto_ass_flag = Optional[bool]=None
    ):
    """
    Aggiorna le liste dei tag assegnati alla query.
    Se fornito un singolo tag viene aggiunto alla lista.
    I parametri con add_ aggiungono alla lista se non già presente, quelli con rmv_ rimuovono se presente
    mentre new_..._list sostituiscono la lista
    """
    

EOF